In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Mounted at /content/drive


In [2]:
data_file = "/content/drive/MyDrive/disseration/dft-road-casualty-statistics-collision-last-5-years.csv"
# Load the dataset
df = pd.read_csv(data_file, low_memory=False)
df.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2021170H10421,2021,170H10421,447098.0,532997.0,-1.270905,54.689833,17,3,2,...,0,0,2,1,2,E01011959,-1,0,0.293588,0.706412
1,2021170H11231,2021,170H11231,450486.0,533118.0,-1.218333,54.690592,17,3,2,...,0,0,1,2,2,E01011973,-1,0,0.017448,0.982552
2,2020170M11750,2020,170M11750,449694.0,519733.0,-1.232884,54.570397,17,3,2,...,0,0,1,1,2,E01012092,-1,0,0.128730,0.871270
3,2021170M31761,2021,170M31761,449744.0,514217.0,-1.233040,54.520825,17,3,1,...,0,0,2,1,2,E01032553,-1,0,0.182698,0.817302
4,2021170S10441,2021,170S10441,445971.0,520834.0,-1.290292,54.580641,17,3,3,...,0,13,2,1,1,E01012258,-1,0,0.016094,0.983906


In [3]:
#size of dataset
rows, columns = df.shape

print("Total number of collision records:", rows)
print("Total number of columns:", columns)

Total number of collision records: 503475
Total number of columns: 44


In [4]:
# List all column names
for col in df.columns:
    print(col)

collision_index
collision_year
collision_ref_no
location_easting_osgr
location_northing_osgr
longitude
latitude
police_force
collision_severity
number_of_vehicles
number_of_casualties
date
day_of_week
time
local_authority_district
local_authority_ons_district
local_authority_highway
local_authority_highway_current
first_road_class
first_road_number
road_type
speed_limit
junction_detail_historic
junction_detail
junction_control
second_road_class
second_road_number
pedestrian_crossing_human_control_historic
pedestrian_crossing_physical_facilities_historic
pedestrian_crossing
light_conditions
weather_conditions
road_surface_conditions
special_conditions_at_site
carriageway_hazards_historic
carriageway_hazards
urban_or_rural_area
did_police_officer_attend_scene_of_accident
trunk_road_flag
lsoa_of_accident_location
enhanced_severity_collision
collision_injury_based
collision_adjusted_severity_serious
collision_adjusted_severity_slight


In [6]:
# I am cleaning column names so Python does not give errors because of spaces or capital letters

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Column names cleaned")
df.columns

Column names cleaned


Index(['collision_index', 'collision_year', 'collision_ref_no',
       'location_easting_osgr', 'location_northing_osgr', 'longitude',
       'latitude', 'police_force', 'collision_severity', 'number_of_vehicles',
       'number_of_casualties', 'date', 'day_of_week', 'time',
       'local_authority_district', 'local_authority_ons_district',
       'local_authority_highway', 'local_authority_highway_current',
       'first_road_class', 'first_road_number', 'road_type', 'speed_limit',
       'junction_detail_historic', 'junction_detail', 'junction_control',
       'second_road_class', 'second_road_number',
       'pedestrian_crossing_human_control_historic',
       'pedestrian_crossing_physical_facilities_historic',
       'pedestrian_crossing', 'light_conditions', 'weather_conditions',
       'road_surface_conditions', 'special_conditions_at_site',
       'carriageway_hazards_historic', 'carriageway_hazards',
       'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident',
 

In [7]:
# I am making a copy of the dataset.
# This means the original df will stay safe.

collision = df.copy()

print("Working copy created")
print("Rows:", collision.shape[0])
print("Columns:", collision.shape[1])

Working copy created
Rows: 503475
Columns: 44


In [8]:
import os

# All Stage 2 output files will be saved here in Google Drive

output_folder = "/content/drive/MyDrive/disseration/stage2_outputs"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print("Stage 2 output folder is ready:")
print(output_folder)

Stage 2 output folder is ready:
/content/drive/MyDrive/disseration/stage2_outputs


In [9]:
# These are the main columns I need for Stage 2 cleaning

important_columns = [
    "collision_index",
    "accident_year",
    "date",
    "time",
    "collision_severity",
    "longitude",
    "latitude",
    "location_easting_osgr",
    "location_northing_osgr",
    "local_authority_highway_current",
    "road_type",
    "weather_conditions",
    "light_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
    "speed_limit"
]

for col in important_columns:
    if col in collision.columns:
        print(col, "found")
    else:
        print(col, "NOT found")

collision_index found
accident_year NOT found
date found
time found
collision_severity found
longitude found
latitude found
location_easting_osgr found
location_northing_osgr found
local_authority_highway_current found
road_type found
weather_conditions found
light_conditions found
road_surface_conditions found
urban_or_rural_area found
speed_limit found


In [10]:
# I am creating a simple cleaning log.
# This will help explain every cleaning decision in the dissertation methodology.

cleaning_log = []

def add_log(step, decision, reason, rows_affected):
    cleaning_log.append({
        "step": step,
        "decision": decision,
        "reason": reason,
        "rows_affected": rows_affected
    })

add_log(
    step="Loaded dataset",
    decision="Loaded the STATS19 collision last five years dataset from Google Drive",
    reason="This is the main dataset for the dissertation project",
    rows_affected=collision.shape[0]
)

print("Cleaning log started")

Cleaning log started


In [11]:
# I am not deleting records where local authority highway is missing.
# These records still contain useful collision information.
# So I will replace missing values with Unknown.

missing_before = collision["local_authority_highway_current"].isna().sum()

collision["local_authority_highway_current"] = collision["local_authority_highway_current"].fillna("Unknown")

missing_after = collision["local_authority_highway_current"].isna().sum()

add_log(
    step="Filled missing local_authority_highway_current",
    decision="Missing values were replaced with Unknown",
    reason="The records are still useful and should not be deleted",
    rows_affected=missing_before
)

print("Missing before:", missing_before)
print("Missing after:", missing_after)

Missing before: 80
Missing after: 0


In [12]:
# Coordinates must be numeric for mapping later.
# I am not guessing or filling missing coordinates.

coordinate_cols = [
    "longitude",
    "latitude",
    "location_easting_osgr",
    "location_northing_osgr"
]

print("Missing coordinate values before conversion:")
print(collision[coordinate_cols].isna().sum())

for col in coordinate_cols:
    collision[col] = pd.to_numeric(collision[col], errors="coerce")

print("\nMissing coordinate values after conversion:")
print(collision[coordinate_cols].isna().sum())

add_log(
    step="Converted coordinate columns",
    decision="Longitude, latitude, easting and northing were converted to numeric",
    reason="Numeric coordinates are needed for later mapping and spatial analysis",
    rows_affected=collision.shape[0]
)

Missing coordinate values before conversion:
longitude                 65
latitude                  65
location_easting_osgr     65
location_northing_osgr    65
dtype: int64

Missing coordinate values after conversion:
longitude                 65
latitude                  65
location_easting_osgr     65
location_northing_osgr    65
dtype: int64


In [13]:
# A record is valid for spatial analysis only if all coordinate columns are available.
# I am keeping missing coordinate rows in the full dataset,
# but they will be removed from the spatial dataset later.

collision["has_valid_coordinates"] = collision[coordinate_cols].notna().all(axis=1)

valid_count = collision["has_valid_coordinates"].sum()
invalid_count = collision.shape[0] - valid_count

add_log(
    step="Created has_valid_coordinates",
    decision="Created a True/False column for coordinate quality",
    reason="Only records with valid coordinates can be mapped later",
    rows_affected=invalid_count
)

collision["has_valid_coordinates"].value_counts()

,count
has_valid_coordinates,
True,503410
False,65


In [14]:
# These are the records that cannot be mapped later

missing_coordinate_records = collision[collision["has_valid_coordinates"] == False]

print("Records with missing coordinates:", missing_coordinate_records.shape[0])

missing_coordinate_records[
    ["collision_index", "date", "longitude", "latitude",
     "location_easting_osgr", "location_northing_osgr"]
].head()

Records with missing coordinates: 65


,collision_index,date,longitude,latitude,location_easting_osgr,location_northing_osgr
463,2021122100128,16/01/2021,NaN,NaN,NaN,NaN
2598,2023122301278,08/06/2023,NaN,NaN,NaN,NaN
6935,2022401282756,01/12/2022,NaN,NaN,NaN,NaN
20543,2021480350825,18/12/2021,NaN,NaN,NaN,NaN
26031,202260B074252,23/05/2022,NaN,NaN,NaN,NaN


In [15]:
# Converting date into proper date format.
# dayfirst=True is used because UK dates usually follow day/month/year format.

missing_dates_before = collision["date"].isna().sum()

collision["date"] = pd.to_datetime(
    collision["date"],
    errors="coerce",
    dayfirst=True
)

missing_dates_after = collision["date"].isna().sum()

add_log(
    step="Converted date",
    decision="Date column was converted into datetime format",
    reason="This is needed for month, day and trend analysis",
    rows_affected=missing_dates_after
)

print("Missing dates before:", missing_dates_before)
print("Missing dates after conversion:", missing_dates_after)

collision[["date"]].head()

Missing dates before: 0
Missing dates after conversion: 0


,date
0,2021-05-22
1,2021-10-20
2,2020-12-01
3,2021-12-09
4,2021-04-12


In [16]:
# Creating time-related columns from the date.
# These will be useful in Stage 3 descriptive analysis.

collision["month"] = collision["date"].dt.month
collision["month_name"] = collision["date"].dt.month_name()
collision["day_name"] = collision["date"].dt.day_name()

add_log(
    step="Created date-based columns",
    decision="Created month, month_name and day_name",
    reason="These columns will help analyse collision trends by month and day",
    rows_affected=collision.shape[0]
)

collision[["date", "month", "month_name", "day_name"]].head()

,date,month,month_name,day_name
0,2021-05-22,5,May,Saturday
1,2021-10-20,10,October,Wednesday
2,2020-12-01,12,December,Tuesday
3,2021-12-09,12,December,Thursday
4,2021-04-12,4,April,Monday


In [17]:
# Converting time and extracting hour.
# Some time values may be missing, so errors='coerce' is used.

time_text = collision["time"].astype("string").str.strip()

time_converted = pd.to_datetime(
    time_text,
    format="%H:%M",
    errors="coerce"
)

collision["time_cleaned"] = time_converted.dt.strftime("%H:%M")
collision["hour"] = time_converted.dt.hour

add_log(
    step="Converted time",
    decision="Time column was cleaned and hour was extracted",
    reason="Hour is needed to analyse daily collision patterns",
    rows_affected=collision["hour"].isna().sum()
)

print("Missing original time values:", collision["time"].isna().sum())
print("Missing hour values after conversion:", collision["hour"].isna().sum())

collision[["time", "time_cleaned", "hour"]].head()

Missing original time values: 0
Missing hour values after conversion: 0


,time,time_cleaned,hour
0,22:44,22:44,22
1,15:50,15:50,15
2,18:00,18:00,18
3,16:55,16:55,16
4,09:02,09:02,9


In [18]:
# Creating a simple weekday/weekend column.
# This helps compare collision patterns during working days and weekends.

collision["weekend_or_weekday"] = np.where(
    collision["day_name"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday"
)

# If date was missing, day_name will also be missing, so I keep that as Unknown
collision.loc[collision["day_name"].isna(), "weekend_or_weekday"] = "Unknown"

add_log(
    step="Created weekend_or_weekday",
    decision="Created Weekday, Weekend and Unknown categories",
    reason="This is useful for comparing collision patterns by type of day",
    rows_affected=collision.shape[0]
)

collision[["date", "day_name", "weekend_or_weekday"]].head()

,date,day_name,weekend_or_weekday
0,2021-05-22,Saturday,Weekend
1,2021-10-20,Wednesday,Weekday
2,2020-12-01,Tuesday,Weekday
3,2021-12-09,Thursday,Weekday
4,2021-04-12,Monday,Weekday


In [19]:
# Collision severity codes:
# 1 = Fatal
# 2 = Serious
# 3 = Slight

severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight",
    -1: "Unknown",
    9: "Unknown"
}

collision["severity_label"] = collision["collision_severity"].map(severity_map).fillna("Unknown")

# Higher number means more severe collision
severity_score_map = {
    "Fatal": 3,
    "Serious": 2,
    "Slight": 1,
    "Unknown": 0
}

collision["severity_score"] = collision["severity_label"].map(severity_score_map)

add_log(
    step="Decoded collision severity",
    decision="Created severity_label and severity_score",
    reason="Readable severity values are easier to analyse and explain",
    rows_affected=collision.shape[0]
)

collision[["collision_severity", "severity_label", "severity_score"]].head()

,collision_severity,severity_label,severity_score
0,3,Slight,1
1,3,Slight,1
2,3,Slight,1
3,3,Slight,1
4,3,Slight,1


In [20]:
# Creating simple 0/1 columns for severity.
# These will be useful later for summaries, dashboard filters and modelling.

collision["fatal_collision"] = np.where(collision["severity_label"] == "Fatal", 1, 0)
collision["serious_collision"] = np.where(collision["severity_label"] == "Serious", 1, 0)
collision["slight_collision"] = np.where(collision["severity_label"] == "Slight", 1, 0)

add_log(
    step="Created severity indicators",
    decision="Created fatal_collision, serious_collision and slight_collision",
    reason="Binary columns make later analysis and modelling easier",
    rows_affected=collision.shape[0]
)

collision[
    ["severity_label", "fatal_collision", "serious_collision", "slight_collision"]
].head()

,severity_label,fatal_collision,serious_collision,slight_collision
0,Slight,0,0,1
1,Slight,0,0,1
2,Slight,0,0,1
3,Slight,0,0,1
4,Slight,0,0,1


In [21]:
# Road type codes are changed into readable labels.

road_type_map = {
    1: "Roundabout",
    2: "One way street",
    3: "Dual carriageway",
    6: "Single carriageway",
    7: "Slip road",
    9: "Unknown",
    12: "One way street / slip road",
    -1: "Unknown"
}

collision["road_type_label"] = collision["road_type"].map(road_type_map).fillna("Unknown")

add_log(
    step="Decoded road type",
    decision="Road type codes were changed into readable labels",
    reason="Road type is important for understanding collision patterns",
    rows_affected=collision.shape[0]
)

collision[["road_type", "road_type_label"]].head()

,road_type,road_type_label
0,6,Single carriageway
1,6,Single carriageway
2,6,Single carriageway
3,6,Single carriageway
4,3,Dual carriageway


In [22]:
# Weather condition codes are changed into readable labels.

weather_map = {
    1: "Fine no high winds",
    2: "Raining no high winds",
    3: "Snowing no high winds",
    4: "Fine high winds",
    5: "Raining high winds",
    6: "Snowing high winds",
    7: "Fog or mist",
    8: "Other",
    9: "Unknown",
    -1: "Unknown"
}

collision["weather_label"] = collision["weather_conditions"].map(weather_map).fillna("Unknown")

add_log(
    step="Decoded weather conditions",
    decision="Weather condition codes were changed into readable labels",
    reason="Weather is useful for analysing collision conditions",
    rows_affected=collision.shape[0]
)

collision[["weather_conditions", "weather_label"]].head()

,weather_conditions,weather_label
0,1,Fine no high winds
1,1,Fine no high winds
2,1,Fine no high winds
3,1,Fine no high winds
4,1,Fine no high winds


In [23]:
# Light condition codes are changed into readable labels.

light_map = {
    1: "Daylight",
    4: "Darkness - lights lit",
    5: "Darkness - lights unlit",
    6: "Darkness - no lighting",
    7: "Darkness - lighting unknown",
    -1: "Unknown"
}

collision["light_label"] = collision["light_conditions"].map(light_map).fillna("Unknown")

add_log(
    step="Decoded light conditions",
    decision="Light condition codes were changed into readable labels",
    reason="Light conditions are useful for understanding day and night collision patterns",
    rows_affected=collision.shape[0]
)

collision[["light_conditions", "light_label"]].head()

,light_conditions,light_label
0,6,Darkness - no lighting
1,1,Daylight
2,4,Darkness - lights lit
3,4,Darkness - lights lit
4,1,Daylight


In [24]:
# Road surface codes are changed into readable labels.

surface_map = {
    1: "Dry",
    2: "Wet or damp",
    3: "Snow",
    4: "Frost or ice",
    5: "Flood over 3cm deep",
    6: "Oil or diesel",
    7: "Mud",
    9: "Unknown",
    -1: "Unknown"
}

collision["road_surface_label"] = collision["road_surface_conditions"].map(surface_map).fillna("Unknown")

add_log(
    step="Decoded road surface conditions",
    decision="Road surface codes were changed into readable labels",
    reason="Road surface is useful for analysing dry, wet or icy road conditions",
    rows_affected=collision.shape[0]
)

collision[["road_surface_conditions", "road_surface_label"]].head()

,road_surface_conditions,road_surface_label
0,1,Dry
1,1,Dry
2,1,Dry
3,2,Wet or damp
4,1,Dry


In [26]:
# Urban/rural codes are changed into readable labels.

urban_rural_map = {
    1: "Urban",
    2: "Rural",
    3: "Unallocated",
    -1: "Unknown"
}

collision["urban_rural_label"] = collision["urban_or_rural_area"].map(urban_rural_map).fillna("Unknown")

add_log(
    step="Decoded urban/rural area",
    decision="Urban/rural codes were changed into readable labels",
    reason="This helps compare urban and rural collision patterns",
    rows_affected=collision.shape[0]
)

collision[["urban_or_rural_area", "urban_rural_label"]].head()

,urban_or_rural_area,urban_rural_label
0,2,Rural
1,1,Urban
2,1,Urban
3,2,Rural
4,2,Rural


In [27]:
# Speed limit is converted to numeric.
# I am not guessing missing or unknown speed limits.

collision["speed_limit"] = pd.to_numeric(collision["speed_limit"], errors="coerce")

# In STATS19, -1 usually means missing or unknown.
collision["speed_limit_cleaned"] = collision["speed_limit"].replace(-1, np.nan)

collision["speed_limit_group"] = collision["speed_limit_cleaned"].astype("string").fillna("Unknown")

add_log(
    step="Cleaned speed limit",
    decision="Speed limit was converted to numeric and grouped as readable text",
    reason="Speed limit is useful for later analysis and modelling",
    rows_affected=collision.shape[0]
)

collision[["speed_limit", "speed_limit_cleaned", "speed_limit_group"]].head()

,speed_limit,speed_limit_cleaned,speed_limit_group
0,60,60.0,60.0
1,30,30.0,30.0
2,20,20.0,20.0
3,40,40.0,40.0
4,70,70.0,70.0


In [28]:
# Checking duplicate collision IDs.
# I am not expecting duplicates because Stage 1 showed 0 duplicate collision IDs.

duplicate_collision_ids = collision["collision_index"].duplicated().sum()

add_log(
    step="Checked duplicate collision IDs",
    decision="Duplicate collision_index values were checked",
    reason="Each collision record should have a unique collision index",
    rows_affected=duplicate_collision_ids
)

print("Duplicate collision IDs:", duplicate_collision_ids)

Duplicate collision IDs: 0


In [29]:
# Checking remaining missing values after cleaning.
# Some missing values may remain because we are not guessing coordinates or other unknown values.

missing_after_cleaning = collision.isna().sum()
missing_after_cleaning = missing_after_cleaning[missing_after_cleaning > 0].sort_values(ascending=False)

missing_after_cleaning.head(20)

,0
location_easting_osgr,65
location_northing_osgr,65
longitude,65
latitude,65
speed_limit_cleaned,15


In [30]:
# Quick check of the new readable columns

label_columns = [
    "severity_label",
    "road_type_label",
    "weather_label",
    "light_label",
    "road_surface_label",
    "urban_rural_label",
    "weekend_or_weekday"
]

for col in label_columns:
    print("\n" + col)
    print(collision[col].value_counts(dropna=False).head(10))


severity_label
severity_label
Slight     386007
Serious    109977
Fatal        7491
Name: count, dtype: int64

road_type_label
road_type_label
Single carriageway    365844
Dual carriageway       74715
Roundabout             29963
Unknown                13113
One way street         11118
Slip road               8722
Name: count, dtype: int64

weather_label
weather_label
Fine no high winds       400956
Raining no high winds     57187
Other                     15270
Unknown                   14617
Raining high winds         5958
Fine high winds            5158
Fog or mist                2296
Snowing no high winds      1741
Snowing high winds          292
Name: count, dtype: int64

light_label
light_label
Daylight                       359837
Darkness - lights lit          104155
Darkness - no lighting          26591
Darkness - lighting unknown      9202
Darkness - lights unlit          3671
Unknown                            19
Name: count, dtype: int64

road_surface_label
road_surface_l

In [31]:
# Full cleaned dataset keeps all records.
# This is important because non-spatial analysis can still use records without coordinates.

collision_cleaned_full = collision.copy()

full_file = output_folder + "/collision_cleaned_full.csv"

collision_cleaned_full.to_csv(full_file, index=False)

add_log(
    step="Saved full cleaned dataset",
    decision="All records were kept in the full cleaned dataset",
    reason="Records without coordinates can still be used for non-spatial analysis",
    rows_affected=collision_cleaned_full.shape[0]
)

print("Full cleaned dataset saved")
print("Rows:", collision_cleaned_full.shape[0])
print("File saved at:", full_file)

Full cleaned dataset saved
Rows: 503475
File saved at: /content/drive/MyDrive/disseration/stage2_outputs/collision_cleaned_full.csv


In [32]:
# Spatial cleaned dataset keeps only records with valid coordinates.
# The 65 records missing coordinates are removed only from this dataset.

collision_cleaned_spatial = collision[collision["has_valid_coordinates"] == True].copy()

spatial_file = output_folder + "/collision_cleaned_spatial.csv"

collision_cleaned_spatial.to_csv(spatial_file, index=False)

removed_from_spatial = collision_cleaned_full.shape[0] - collision_cleaned_spatial.shape[0]

add_log(
    step="Saved spatial cleaned dataset",
    decision="Records missing coordinates were excluded only from the spatial dataset",
    reason="Records without coordinates cannot be mapped or assigned to a future 500m grid",
    rows_affected=removed_from_spatial
)

print("Spatial cleaned dataset saved")
print("Rows:", collision_cleaned_spatial.shape[0])
print("Removed from spatial dataset:", removed_from_spatial)
print("File saved at:", spatial_file)

Spatial cleaned dataset saved
Rows: 503410
Removed from spatial dataset: 65
File saved at: /content/drive/MyDrive/disseration/stage2_outputs/collision_cleaned_spatial.csv


In [33]:
# Creating a short summary of Stage 2 outputs and main cleaning results.

stage2_summary = pd.DataFrame({
    "item": [
        "Raw records",
        "Raw columns",
        "Full cleaned records",
        "Spatial cleaned records",
        "Records removed only from spatial dataset",
        "Duplicate collision IDs",
        "Missing longitude",
        "Missing latitude",
        "Missing easting",
        "Missing northing",
        "Valid coordinate records",
        "Invalid coordinate records",
        "Missing local_authority_highway_current after cleaning"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        collision_cleaned_full.shape[0],
        collision_cleaned_spatial.shape[0],
        removed_from_spatial,
        duplicate_collision_ids,
        collision["longitude"].isna().sum(),
        collision["latitude"].isna().sum(),
        collision["location_easting_osgr"].isna().sum(),
        collision["location_northing_osgr"].isna().sum(),
        collision["has_valid_coordinates"].sum(),
        collision.shape[0] - collision["has_valid_coordinates"].sum(),
        collision["local_authority_highway_current"].isna().sum()
    ]
})

summary_file = output_folder + "/stage2_cleaning_summary.csv"

stage2_summary.to_csv(summary_file, index=False)

print("Stage 2 summary saved")
print("File saved at:", summary_file)

stage2_summary

Stage 2 summary saved
File saved at: /content/drive/MyDrive/disseration/stage2_outputs/stage2_cleaning_summary.csv


,item,value
0,Raw records,503475
1,Raw columns,44
2,Full cleaned records,503475
3,Spatial cleaned records,503410
4,Records removed only from spatial dataset,65
5,Duplicate collision IDs,0
6,Missing longitude,65
7,Missing latitude,65
8,Missing easting,65
9,Missing northing,65


In [34]:
# Final check of all Stage 2 output files

print("Files created in Stage 2 folder:")

for file in os.listdir(output_folder):
    print("-", file)

print("\nFull cleaned dataset shape:")
print(collision_cleaned_full.shape)

print("\nSpatial cleaned dataset shape:")
print(collision_cleaned_spatial.shape)

print("\nCoordinate validity:")
print(collision_cleaned_full["has_valid_coordinates"].value_counts())

print("\nSeverity distribution:")
print(collision_cleaned_full["severity_label"].value_counts())

Files created in Stage 2 folder:
- collision_cleaned_full.csv
- collision_cleaned_spatial.csv
- stage2_cleaning_summary.csv

Full cleaned dataset shape:
(503475, 63)

Spatial cleaned dataset shape:
(503410, 63)

Coordinate validity:
has_valid_coordinates
True     503410
False        65
Name: count, dtype: int64

Severity distribution:
severity_label
Slight     386007
Serious    109977
Fatal        7491
Name: count, dtype: int64
